# 27200 · Multi-omics integration with MOFA
## Exercise B: break the model, compare models, critique the AI

**Group:** _____ **Names:** _____

*Exercise by Carsten; Python version of his R workbook for Google Colab.*

## How to use this notebook

* Same rules as Exercise A: run cells in order, edit only lines marked **`# DECIDE`**,
  answer the **Q**s in the text cells.
* The setup cell asks you to upload `27200_mofa_data.zip` again: this notebook runs on its own machine.
* Part 2 is a controlled experiment: you change **one** thing, **predict** the consequence,
  then check it. Write the prediction *before* you run the model.

| Time      | Activity                                                 |
|-----------|----------------------------------------------------------|
| 0–5 min   | §1 Setup: rebuild your Exercise A model                  |
| 5–45 min  | §2 Stress tests: break the model on purpose              |
| 45–60 min | §3 Model comparison: PCA and sparse PLS-DA               |
| 60–85 min | §4 AI-assisted interpretation and critique               |
| 85–90 min | §5 Notes for the one-minute paper                        |

## 1 · Setup

In [ ]:
# Run once per session: installs MOFA's Python engine (mofapy2, about 30 s), downloads the
# helper functions and asks you to upload the data: 27200_mofa_data.zip from DTU Learn (Week 10)
!pip install -q mofapy2==0.7.5

import os, json, urllib.request
if not os.path.exists("mofa_helpers.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/UadKLab/27200_Data-Driven-Bioengineering/main/weeks/week10_multiomics_integration/notebooks/mofa_helpers.py", "mofa_helpers.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mofa_helpers import *          # build_views(), fit_mofa(), the plot functions, ...

mc = load_metacardis("data")        # one dict: metadata, drugs, microbiome, serum, taxonomy

In [ ]:
# Paste the settings printed at the end of Exercise A here.                      # DECIDE
# Left empty, the notebook uses models/settings_exerciseA.json (only found when both notebooks
# run in the same folder, e.g. on your own computer) or the default settings.
my_settings = {}

if not my_settings and os.path.exists("models/settings_exerciseA.json"):
    my_settings = json.load(open("models/settings_exerciseA.json"))
settings = merge_settings(my_settings)
print("Using", "your Exercise A settings" if my_settings else "the default settings")

# Choose which covariates to compare later in the analysis: clinical factors, drugs
covariates = ["Age", "Gender", "Nationality", "Antibiotics", "phenotype",
              "Metformin", "Statin", "Aspirine", "PPI"]                          # DECIDE

In [ ]:
# All fitted models are kept in one dict; the baseline is your Exercise A model.
fits = {"baseline": fit_mofa(build_views(mc, settings), settings)}

## 2 · Stress tests: break the model on purpose

Each scenario changes **one** aspect of the data or model and keeps everything else as in
your baseline. Your group should choose one or two scenarios.

| Scenario | What changes | Assumption / failure mode under test |
|----------|--------------|--------------------------------------|
| `S1_compositionality` | microbiome entered as raw relative abundance (no log-ratio) | compositional, zero-inflated data vs Gaussian likelihood |
| `S2_no_scaling` | no log-transform, no z-scoring, no view scaling | features and views on different scales |
| `S3_batch` | simulated plate effect on the metabolome; CAD samples mostly on plate 2 | technical factor confounded with phenotype |
| `S3b_batch_removed` | as S3, but plate means removed before MOFA | the price of correcting a confounded batch |
| `S4_scrambled` | microbiome rows shuffled across all samples | sample linkage (same individual in every view) |
| `S4b_partly_scrambled` | 20 % of microbiome samples shuffled | a realistic sample mix-up |
| `S5_drugs_in_serum` | the measured drugs and drug metabolites are kept in the serum view | the same exposure measured twice |
| `S5b_no_drug_view` | drug view removed | does removing a view remove its confounding? |
| `S6_gaussian_drugs` | Gaussian instead of Bernoulli likelihood for drugs | likelihood specification |
| `S7_country_groups` | multi-group MOFA with one group per country | centre effects vs phenotype confounding |

In [ ]:
scenarios = dict(
    S1_compositionality  = dict(mb_transform="none"),
    S2_no_scaling        = dict(clinical_log=[], clinical_scale=False, met_scale=False, scale_views=False),
    S3_batch             = dict(batch_effect=0.5, batch_confounding=0.8),
    S3b_batch_removed    = dict(batch_effect=0.5, batch_confounding=0.8, batch_correct=True),
    S4_scrambled         = dict(scramble_view="microbiome", scramble_fraction=1),
    S4b_partly_scrambled = dict(scramble_view="microbiome", scramble_fraction=0.2),
    S5_drugs_in_serum    = dict(met_drop_xenobiotics=False),
    S5b_no_drug_view     = dict(views=[v for v in settings["views"] if v != "drugs"]),
    S6_gaussian_drugs    = dict(drug_likelihood="gaussian"),
    S7_country_groups    = dict(group_by="Nationality"),
)
# You may change the strength of a scenario (e.g. batch_effect, scramble_fraction).

my_scenarios = ["S1_compositionality", "S2_no_scaling"]                           # DECIDE

**Q1 – Prediction (before running!).** For each of your scenarios, predict what will
happen to (a) the variance explained per view, (b) the correspondence between baseline and
new factors, (c) the association of factors with phenotype.

> *Prediction:*

In [ ]:
for sc in my_scenarios:
    st = merge_settings(settings, scenarios[sc])   # baseline settings + this scenario's changes
    print(f"--- {sc}")
    fits[sc] = fit_mofa(build_views(mc, st), st)

### Diagnostics

In [ ]:
# (a) Variance explained per factor and view (values >= 0.1 % printed)
plot_r2_compare({k: fits[k] for k in ["baseline"] + my_scenarios})

In [ ]:
# (b) Which baseline factors survive? |correlation| between baseline and new factor values
fig, axes = plt.subplots(1, len(my_scenarios), figsize=(7 * len(my_scenarios), 5), squeeze=False)
for ax, sc in zip(axes[0], my_scenarios):
    plot_factor_match(fits["baseline"], fits[sc], ("baseline", sc), ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# (c) Factor ~ covariate association; 'plate' is added automatically for S3 scenarios
for k in ["baseline"] + my_scenarios:
    cols = [c for c in covariates + ["plate"] if c in fits[k].metadata.columns]
    plot_covariate_r2(factor_covariates(fits[k], cols), k); plt.show()

> **Caution.** `plot_top_weights()` scales weights to −1…1 *within* a factor. A factor that
> explains ~0 % of a view still shows "top" features for that view; they are noise.
> Check the factor's R² for that view in the plot above before reading its weights.

In [ ]:
# Zoom in on one factor of one scenario
zoom_model  = my_scenarios[0]                                                     # DECIDE
zoom_factor = 1                                                                   # DECIDE
zoom_view   = "microbiome"   # "clinical", "microbiome", "drugs", "metabolome"    # DECIDE

plot_top_weights(fits[zoom_model], view=zoom_view, factor=zoom_factor, nfeatures=10)
plot_factor(fits[zoom_model], factors=[zoom_factor], group_by="phenotype", color_by="Nationality")

<details><summary><b>Hints per scenario (open only if stuck)</b></summary>

* **S1**: Compare the microbiome column of the R² plot. Look at the histogram of raw relative
  abundances in Exercise A §3.
* **S2**: Zoom into the clinical view of the main factor. Which markers carry the weight now,
  and what do they have in common?
* **S3 / S3b**: Compare the phenotype R² of the main factor in baseline, S3 and S3b. Why does
  removing the plate effect also remove something else?
* **S4 / S4b**: Do the microbiome factors still associate with *any* covariate? Which shared
  baseline factor lost its microbiome partner? Would you have noticed S4b without the baseline?
* **S5 / S5b**: What happens to the drug R²? In S5b, run the medication check below: is the
  Metformin signal gone when the drug view is gone?
* **S6**: Are the factors different? Why can the drug R² change while the factors do not?
* **S7**: What happens to the Nationality *and* the phenotype association? MOFA centres each
  group (country) separately before training.
</details>

### Medication as a confounder (all groups)

Does a factor that separates patients from controls also vary with medication *among
patients with the same diagnosis*? If so, part of the "disease" signal may be drug signal.

In [ ]:
check_model = "baseline"   # DECIDE: e.g. "baseline" or "S5b_no_drug_view" (if fitted)
stratum     = "CAD"        # DECIDE: phenotype to look within: T2D, CAD, Obesity
exposure    = "PPI"        # DECIDE: drug to test within that phenotype: Metformin, Statin, Aspirine, PPI

res = factor_covariates_within(fits[check_model], exposure, "phenotype", stratum)
display(res[res.p_adj < 0.05])   # if any factor is significant, plot it below

plot_factor(fits[check_model], factors=[1, 2], group_by="phenotype", color_by=exposure)

**Q2.** For each scenario: what happened, which assumption was violated, and did the result
match your prediction?

> *Answer:*

**Q3.** In a real analysis you have no "baseline" to compare against. Look at the different scenarios and name one
diagnostic you could run on a *single* model to detect confounders and verify assumptions (e.g. a permutation, a
negative control, a covariate you would always check).

> *Answer:*

**Q4.** Within one phenotype, is your main factor associated with a drug? What does that
imply for a sentence like "Factor 1 captures cardiometabolic disease"? Patients with more severe
disease are often prescribed more drugs, so a drug association within one diagnosis can also
reflect disease severity rather than the drug itself (confounding by indication). How could you
tell the two apart?

> *Answer:*

## 3 · Model comparison: PCA and sparse PLS-DA

Same data, different model classes. PCA and PLS-DA need one complete matrix, so views are
z-scored, missing values set to the feature mean, each view down-weighted by
√(number of features), and all views concatenated.

In [ ]:
views = build_views(mc, settings)
X = block_matrix(views)                     # samples x (all features of all views)

from sklearn.decomposition import PCA
pca = PCA(n_components=10).fit(X)
pcs = pd.DataFrame(pca.transform(X), index=X.index, columns=[f"PC{i + 1}" for i in range(10)])

In [ ]:
# Which view do the leading PCs come from? Share of squared loadings per view
loadings = pd.DataFrame(pca.components_.T ** 2, index=X.columns, columns=pcs.columns)
loadings.groupby(X.columns.str.split(":").str[0]).sum().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_factor_match(fits["baseline"], pcs, ("MOFA factors", "principal components"), ax=axes[0])
plot_covariate_r2(covariate_r2(pcs, views.covariates, covariates), "PCA", ax=axes[1])
plt.tight_layout(); plt.show()

**Sparse PLS-DA** is *supervised*: it searches for the combination of features that best separates
the phenotype labels. To see what that implies, fit it once on the true labels and once on
randomly permuted labels.

*In Python we approximate sparse PLS-DA (mixOmics `splsda` in R) in two steps: keep the
`keepX × 2` features that best separate the labels (F-test), then fit PLS-DA with two components.*

In [ ]:
y = views.covariates.loc[X.index, "phenotype"].to_numpy()
labels = {"true labels": y,
          "permuted labels": np.random.default_rng(settings["seed"]).permutation(y)}
keepX = 50                                                                        # DECIDE: keepX

for name, lab in labels.items():
    _, scores = splsda_fit(X, lab, ncomp=2, keepX=keepX)
    plot_splsda(scores, lab, name)

In [ ]:
# Cross-validated balanced error rate (5-fold x 3). Chance for 5 classes = 0.80.
# The feature selection is redone inside every training fold.
pd.Series({name: splsda_cv_ber(X, lab, ncomp=2, keepX=keepX) for name, lab in labels.items()}).round(2)

**Q5.** Which views dominate the leading PCs, and why might a view with only a few features
dominate after block scaling? Which MOFA factors have a PCA counterpart and which do not?

> *Answer:*

**Q6.** Compare the sparse PLS-DA score plots for true and permuted labels, then the cross-validated
error. What does a nicely separated sparse PLS-DA plot prove on its own? When would you choose
MOFA, and when sparse PLS-DA (or its multi-block version, DIABLO)?

> *Answer:*

## 4 · AI-assisted interpretation and critique

You will ask an LLM to interpret one factor, then **check every claim** against the data and
the primary literature.

### Export a factor brief

In [ ]:
factor_to_interpret = 1                                                           # DECIDE
brief = llm_brief(fits["baseline"], factor_to_interpret, settings, covariates,
                  file=f"models/llm_brief_factor{factor_to_interpret}.md")

### Prompt the LLM

Copy the prompt below into the LLM and paste the brief (the text printed above, or the
`.md` file in `models/`) where indicated. Do not shorten the constraints.

```text
ROLE
You are assisting a biomedical data analyst. Be precise, conservative and explicit about
uncertainty.

CONTEXT
Below is a summary of one latent factor from a MOFA2 model (unsupervised multi-omics
factor analysis) fitted to a cross-sectional cohort. [PASTE THE BRIEF HERE]

TASK
1. Propose at most three interpretations of this factor. For each, list the specific
   features and numbers from the brief that support it.
2. Label every claim as
   (D) directly visible in the numbers provided,
   (L) based on published literature, or
   (S) speculative mechanism.
3. Make sure to provide at least two of each D, L and S claim.
4. For every (L) claim give one primary reference (authors, year, journal, DOI). If you
   are not certain that a reference exists, write "reference uncertain" instead.
5. List non-biological explanations (medication, recruitment country/centre, how the
   phenotypes are defined, technical artefacts, preprocessing choices) that could produce
   the same pattern, and one analysis that would distinguish each from a biological one.
6. State what this factor does NOT show.

CONSTRAINTS
- Do not invent features, covariates or numbers that are not in the brief.
- Factor weights are relative loadings, not effect sizes; the sign of a factor is arbitrary.
- Metabolites named X.12345 are unannotated; do not assign them an identity.
```

### Check the claims

In [ ]:
# Does the claimed feature really track the factor? Copy exact feature names from the brief.
claim_view     = "microbiome"                                                     # DECIDE
claim_features = ["Prevotella_copri"]                                             # DECIDE
plot_data_scatter(fits["baseline"], view=claim_view, factor=factor_to_interpret,
                  features=claim_features, color_by="phenotype", add_lm=True)

Check literature claims at the source: resolve every DOI (`https://doi.org/<DOI>`), find the
paper on PubMed, and confirm that the paper actually reports what the LLM says it does. The
three MetaCardis papers are a good place to start:
Vieira-Silva *et al.* 2020 (doi:10.1038/s41586-020-2269-x),
Forslund *et al.* 2021 (doi:10.1038/s41586-021-04177-9),
Fromentin *et al.* 2022 (doi:10.1038/s41591-022-01688-4).

**Validation log** (at least five claims, including at least one (L) claim with a reference):

| # | LLM claim (short) | Type (D/L/S) | How we checked | Verdict (accepted / corrected / rejected) | Evidence |
|---|-------------------|--------------|----------------|-------------------------------------------|----------|
| 1 | *LLM claim* | *Type* | *Check* | *Verdict* | *Evidence* |
| 2 | *LLM claim* | *Type* | *Check* | *Verdict* | *Evidence* |
| 3 | *LLM claim* | *Type* | *Check* | *Verdict* | *Evidence* |
| 4 | *LLM claim* | *Type* | *Check* | *Verdict* | *Evidence* |
| 5 | *LLM claim* | *Type* | *Check* | *Verdict* | *Evidence* |

**Q7.** Did the LLM invent a reference, a feature or a taxon–function link? Did it list a
confounder you had not considered? Which part of the prompt helped most, and how would you
change it?

> *Answer:*

## 5 · Notes for the one-minute paper

Draft your group's one-minute paper here (max. ~150 words in total).

* **What we found** (one sentence, with a number):
* **What we assumed** (the two preprocessing/model choices that matter most):
* **What could be wrong** (strongest alternative explanation, and what your stress test showed):
* **How AI was used** (what we accepted, corrected and rejected):